## Apply bigwarp transformation to barcode positions

In [1]:
import os
import numpy as np
import pandas as pd
import h5py
import json

def save_cytassist_coordinates_for_bigwarp(tissue_positions_path, feature_slice_path, cytassist_microns_per_pixel, output_path, bin_size):
    # Create output directory
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Load the tissue positions
    df = pd.read_parquet(tissue_positions_path)

    # Load the metadata and extract affine transformation matrix
    with h5py.File(feature_slice_path, "r") as f5:
        metadata = json.loads(dict(f5.attrs)["metadata_json"])
    affine_matrix = np.array(metadata['transform_matrices']['spot_colrow_to_cytassist_colrow'])

    # Apply the affine transformation matrix
    if bin_size == 2:
        coords = np.hstack([df[['array_col', 'array_row']].values, np.ones((df.shape[0], 1))])
    elif bin_size > 2:
        coords = np.hstack([df[['array_col', 'array_row']].values * (bin_size/2), np.ones((df.shape[0], 1))])
    if bin_size < 2:
        raise ValueError("bin_size must be 2 or greater.")
    
    transformed_coords = coords @ affine_matrix.T

    # Save the transformed coordinates
    df_transformed = pd.DataFrame({
        'x': transformed_coords[:, 0] * (cytassist_microns_per_pixel / 1000), # bigwarp needs the coordinates in millimeters.
        'y': transformed_coords[:, 1] * (cytassist_microns_per_pixel / 1000),
    })
    
    df_transformed.to_csv(output_path, index=False)

In [8]:
for bin_size in [2, 8, 16]:
    bin_name = f'{bin_size:03}um'
    
    save_cytassist_coordinates_for_bigwarp(
        tissue_positions_path = f'E:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/1/SCA002_reseq/COUNTS/outs/binned_outputs/square_{bin_name}/spatial/tissue_positions.parquet', 
        feature_slice_path = f'E:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/1/SCA002_reseq/COUNTS/outs/feature_slice.h5', 
        cytassist_microns_per_pixel = 4.625, 
        output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/1/SCA002_reseq/registration_tests/Bigwarp/transformed_barcode_positions/square_{bin_name}/cytassist_coordinates.csv',
        bin_size = bin_size)

In [23]:
for bin_size in [2, 8, 16]:
    bin_name = f'{bin_size:03}um'
    
    save_cytassist_coordinates_for_bigwarp(
        tissue_positions_path = f'E:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/2/SCA003_reseq/COUNTS/outs/binned_outputs/square_{bin_name}/spatial/tissue_positions.parquet', 
        feature_slice_path = f'E:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/2/SCA003_reseq/COUNTS/outs/feature_slice.h5', 
        cytassist_microns_per_pixel = 4.625, 
        output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/2/SCA003_reseq/registration_tests/Bigwarp/transformed_barcode_positions/square_{bin_name}/cytassist_coordinates.csv',
        bin_size = bin_size)

Use warp_points_from_csv.groovy script in FIJI to calculate transformed coordinates. This script takes a landmark.csv file made in BigWarp and allows transforming coordinates in both directions (i.e. from moving image to fixed image or from fixed image to moving image) based on a variety of different transformation types (including affine and thin-plate spline). Note that this requires the input coordinates to be in millimeters and that the output coordinates will be in millimeters as well.

It seenms best to use the landmarks from slidescanner to cytassist and calculate the inverse. For some reasons, using the landmarks from cytassist to slidescanner gives weird disconnects in the TPS-transformation (the affine transformation looks fine though)

## Convert HE to h5 for ilastik

In [4]:
from skimage import io
import h5py

input = r'e:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\IMAGES\SCA002_HE_hires.tif'
output = r'E:\2024-07-UTBenchmark-SpC\data\processed\Visium_HD\1\SCA002_reseq\registration_tests\harpy\ilastik\SCA002_HE_hires.h5'
img = io.imread(input)

with h5py.File(output, 'w') as h5f:
    h5f.create_dataset('HE', data=img, chunks=(512, 512, 3))

## Convert col/row coordinates for Bigwarp

In [4]:
def save_colrow_coordinates_for_bigwarp(tissue_positions_path, output_path, bin_size):
    # Create output directory
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Load the tissue positions
    df = pd.read_parquet(tissue_positions_path)

    # Apply the affine transformation matrix
    if bin_size == 2:
        coords = np.hstack([df[['array_col', 'array_row']].values, np.ones((df.shape[0], 1))])
    elif bin_size > 2:
        coords = np.hstack([df[['array_col', 'array_row']].values * (bin_size/2), np.ones((df.shape[0], 1))])
    if bin_size < 2:
        raise ValueError("bin_size must be 2 or greater.")
    
    # Save the transformed coordinates
    df_transformed = pd.DataFrame({
        'x': coords[:, 0] * (1 / 1000), # bigwarp needs the coordinates in millimeters. For the transcripts, Bigwarp wrongly assumed that the microns_per_pixel was 1.
        'y': coords[:, 1] * (1 / 1000),
    })
    
    df_transformed.to_csv(output_path, index=False)

In [5]:
for bin_size in [2, 8, 16]:
    bin_name = f'{bin_size:03}um'
    
    save_colrow_coordinates_for_bigwarp(
        tissue_positions_path = f'E:/2024-07-UTBenchmark-SpC/data/raw/Visium_HD/1/SCA002_reseq/COUNTS/outs/binned_outputs/square_{bin_name}/spatial/tissue_positions.parquet', 
        output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Visium_HD/1/SCA002_reseq/registration_tests/Bigwarp/transformed_barcode_positions/square_{bin_name}/colrow_coordinates.csv',
        bin_size = bin_size)